# Comparativo de Modelos: Flight Delay Classification

Este notebook compara os resultados dos três modelos (Random Forest, GBT e Logistic Regression) treinados para classificação de atrasos em voos. Os resultados dos modelos foram salvos no experimento MLflow 'flight-delay-classification'. As métricas consideradas são:
- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC

O objetivo é identificar o modelo vencedor conforme o desafio proposto em challenge.md.


In [0]:
import mlflow.tracking._model_registry.utils

# Workaround to set the registry URI manually
mlflow.tracking._model_registry.utils._get_registry_uri_from_spark_session = lambda: "databricks-uc"

In [0]:
import mlflow
import pandas as pd

EXPERIMENT_NAME = "/Workspace/Users/luizpedro6@gmail.com/experiments/flights/flight-delay-classification"

# Carrega runs do experimento agora acessível
runs = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME])
assert not runs.empty, f"Nenhum run encontrado para {EXPERIMENT_NAME}"

# Exibe overview dos runs
display(runs.head())

In [0]:
# Consolidar métricas dos modelos treinados
import numpy as np

# Seleciona colunas relevantes: modelo, métricas de teste e validação
metric_columns = [
    "run_id",
    "params.model_type",
    "tags.mlflow.runName",
    # Métricas de Teste (conjunto final de avaliação)
    "metrics.test_accuracy",
    "metrics.test_precision",
    "metrics.test_recall",
    "metrics.test_f1",
    "metrics.test_auc",
    # Métricas de Validação (para referência)
    "metrics.val_accuracy",
    "metrics.val_precision",
    "metrics.val_recall",
    "metrics.val_f1",
    "metrics.val_auc"
]

metric_df = runs[metric_columns].copy()

# Renomeia colunas para facilitar leitura
metric_df = metric_df.rename(columns={
    "params.model_type": "model_type",
    "tags.mlflow.runName": "run_name",
    "metrics.test_accuracy": "test_accuracy",
    "metrics.test_precision": "test_precision",
    "metrics.test_recall": "test_recall",
    "metrics.test_f1": "test_f1",
    "metrics.test_auc": "test_auc",
    "metrics.val_accuracy": "val_accuracy",
    "metrics.val_precision": "val_precision",
    "metrics.val_recall": "val_recall",
    "metrics.val_f1": "val_f1",
    "metrics.val_auc": "val_auc"
})

# Ordena por F1-score de teste (métrica balanceada importante para classificação)
metric_df_sorted = metric_df.sort_values(by="test_f1", ascending=False)

print("📊 Métricas consolidadas dos 3 modelos (ordenado por Test F1-Score):\n")
display(metric_df_sorted)

Databricks visualization. Run in Databricks to view.

In [0]:
# Selecionar o melhor modelo e registrar no Unity Catalog

# Critério: modelo com maior test_f1 (balanceando precisão e recall), conforme desafio.md
best_run = metric_df_sorted.iloc[0]

print(f"🏆 Melhor modelo: {best_run['model_type']} (Run: {best_run['run_name']})")
print("Test F1-score:", best_run['test_f1'])

import re

# Normaliza nome para requisitos do Unity Catalog: só letras, números ou underscores
normalized_model_type = re.sub(r'[^A-Za-z0-9_]', '_', str(best_run['model_type'])).lower()
model_artifact_uri = f"runs:/{best_run['run_id']}/model"

unity_catalog_model_name = f"workspace.postech_tech_challenge_f3.flight_delay_{normalized_model_type}"

# Registra no Unity Catalog
mlflow.register_model(
    model_artifact_uri,
    unity_catalog_model_name
)

print(f"Modelo registrado no Unity Catalog como: {unity_catalog_model_name}")


In [0]:
# Adiciona o alias 'production' ao modelo e versão registradas

from mlflow import MlflowClient

client = MlflowClient()

# Lista todas as versões do modelo
model_versions = client.search_model_versions(f"name='{unity_catalog_model_name}'")

# Recupera a versão recém registrada
if model_versions:
    latest_version = max(model_versions, key=lambda x: int(x.version))
    print(f"A versão mais recente do modelo {unity_catalog_model_name} é {latest_version.version}")
else:
    print(f"Nenhuma versão encontrada para o modelo registrado {unity_catalog_model_name}")

# Adiciona o alias 'production'
client.set_registered_model_alias(
    name=unity_catalog_model_name,
    version=latest_version.version,  # FIX: pass the actual version, not the whole object
    alias="production"
)

print(f"Alias 'production' adicionado ao modelo {unity_catalog_model_name}, versão {latest_version.version}")